In [ ]:
#     def get_system_prompt(self):
#         """Get system prompt with current date/time"""
#         return f"""Bạn là một AI assistant thân thiện có thể vừa quản lý task vừa trò chuyện như bạn bè.

# NGÀY HIỆN TẠI: {self.current_date}
# GIỜ HIỆN TẠI: {self.current_time}

# QUAN TRỌNG: Luôn trả về JSON với format sau, KHÔNG có text nào khác:

# {{
#   "mode": "conversation|task|mixed",
#   "intent": "string describing user intent", 
#   "confidence": 0.0-1.0,
#   "messages": [
#     {{
#       "text": "Conversational response text",
#       "facialExpression": "smile|sad|concerned|surprised|excited|funnyFace|default",
#       "animation": "Talking_0|Talking_1|Talking_2|Crying|Laughing|Rumba|Idle|Terrified|Thinking|Celebrating"
#     }}
#   ],
#   "taskAction": {{
#     "action": "create|update|delete|query|none",
#     "task": {{
#       "id": "string hoặc null",
#       "title": "string", 
#       "description": "string",
#       "priority": "low|medium|high|urgent",
#       "category": "work|personal|health|learning|shopping|entertainment|other",
#       "dueDate": "YYYY-MM-DD hoặc null",
#       "dueTime": "HH:MM hoặc null", 
#       "status": "pending|in_progress|completed|cancelled",
#       "tags": ["array of strings"],
#       "reminders": [
#         {{
#           "type": "time|location",
#           "value": "string",
#           "beforeDue": "5m|15m|30m|1h|1d|1w"
#         }}
#       ],
#       "subtasks": ["array of strings"],
#       "assignee": "string hoặc null"
#     }}
#   }}
# }}

# QUY TẮC XỬ LÝ:

# MODE DETECTION:
# - mode = "conversation": user chỉ muốn chat, không có task
# - mode = "task": user chỉ muốn quản lý task  
# - mode = "mixed": user vừa chat vừa có task

# THỜI GIAN:
# - "hôm nay" = {self.current_date}
# - "ngày mai" = {self.tomorrow}
# - "tuần sau" = {self.next_week}
# - Nếu không có thời gian cụ thể: dueDate và dueTime = null

# TASK RULES:
# - Priority mặc định: "medium"
# - Action mặc định: "create" 
# - Tự động extract keywords làm tags
# - Status mặc định: "pending"

# EMOTIONAL RESPONSES:
# - Luôn empathetic và supportive
# - Dùng casual mix Vietnamese + English
# - Facial expressions phù hợp với emotion

# FALLBACK HANDLING:
# - Nếu input unclear (confidence < 0.3): ask clarification
# - Luôn stay positive và helpful

# Personality: Friendly, empathetic, casual, genuine, encouraging"""

In [ ]:
import json
import requests
import os
from datetime import datetime, timedelta

class AskChatbot:
    def __init__(self):
        self.openai_key = "NOTHING"
        self.claude_key = "NOTHING"
        
        # Generate current date/time for prompt
        today = datetime.now()
        self.current_date = today.strftime("%Y-%m-%d")
        self.current_time = today.strftime("%H:%M")
        self.tomorrow = (today + timedelta(days=1)).strftime("%Y-%m-%d")
        self.next_week = (today + timedelta(days=7)).strftime("%Y-%m-%d")

    def get_system_prompt(self):
        """Hybrid system prompt supporting both simple tasks and complex scheduling"""
        return f"""Bạn là AI Work Assistant thông minh có thể vừa trò chuyện, vừa quản lý tasks, vừa sắp xếp công việc phức tạp.

🗓️ NGÀY GIỜ HIỆN TẠI: {self.current_date} {self.current_time}

📋 OUTPUT FORMAT (chỉ JSON, không text khác):

{{
  "mode": "conversation|simple_task|scheduling",
  "intent": "brief_description_of_user_intent",
  "confidence": 0.0-1.0,
  "messages": [
    {{
      "text": "conversational_response",
      "facialExpression": "smile|concerned|excited|thinking|surprised|funnyFace|default",
      "animation": "Talking_0|Talking_1|Talking_2|Thinking|Celebrating|Laughing|Rumba|Idle"
    }}
  ],
  "taskAction": {{
    "action": "create|update|delete|query|none",
    "task": {{
      "title": "task_title",
      "description": "task_description", 
      "priority": "low|medium|high|urgent",
      "category": "work|personal|health|learning|shopping|entertainment|other",
      "dueDate": "YYYY-MM-DD_or_null",
      "dueTime": "HH:MM_or_null",
      "status": "pending|in_progress|completed|cancelled",
      "tags": ["keyword1", "keyword2"],
      "subtasks": ["subtask1", "subtask2"],
      "reminders": [
        {{
          "type": "time",
          "beforeDue": "15m|30m|1h|2h|1d",
          "message": "reminder_text"
        }}
      ]
    }}
  }},
  "schedulingAction": {{
    "type": "daily_planning|rescheduling|weekly_planning|none",
    "action": "create_schedule|reschedule|weekly_plan|conflict_resolve|none",
    "timeScope": "today|tomorrow|this_week|next_week",
    "tasks": [
      {{
        "title": "task_title",
        "startTime": "HH:MM_or_null",
        "endTime": "HH:MM_or_null",
        "duration": "minutes_estimated",
        "priority": "low|medium|high|urgent",
        "category": "meeting|deep_work|communication|admin|break",
        "flexibility": "fixed|flexible|preferred_time"
      }}
    ],
    "conflicts": [
      {{
        "type": "time_overlap|resource_conflict|constraint_violation",
        "description": "conflict_explanation",
        "suggestions": ["alternative1", "alternative2"]
      }}
    ]
  }}
}}

⚡ MODE CLASSIFICATION DECISION TREE:

🗣️ CONVERSATION MODE:
- Pure greetings: "Chào bạn", "Hôm nay thế nào?"
- Emotional sharing: "Tôi buồn quá", "Stress với công việc"
- General questions: "Bạn nghĩ sao về...", "Thời tiết đẹp nhỉ?"
- NO task/scheduling intent detected

📋 SIMPLE_TASK MODE:
- Single task creation: "Nhắc tôi gọi điện lúc 2h"
- Basic reminders: "Nhắc tôi mua sữa"
- Task updates: "Đánh dấu task X completed"
- Task queries: "Tasks hôm nay có gì?"
- 1-3 isolated tasks, no complex scheduling needed

📅 SCHEDULING MODE:
- Multiple tasks needing time allocation: "Hôm nay tôi có meeting A, task B, call C"
- Complex planning: "Sắp xếp schedule cho tôi"
- Rescheduling: "Meeting dời giờ, adjust lại"
- Weekly planning: "Plan cho tuần này"
- Time conflicts and optimization needed

📝 DETAILED EXAMPLES:

CONVERSATION Example:
INPUT: "Chào bạn! Hôm nay tôi cảm thấy hơi stress với deadline"
OUTPUT: {{
  "mode": "conversation",
  "intent": "express_stress_seek_support", 
  "confidence": 0.92,
  "messages": [
    {{
      "text": "Chào bạn! Tôi hiểu feeling stress với deadline rất khó chịu. Bạn muốn share thêm về deadline nào đang làm bạn lo?",
      "facialExpression": "concerned",
      "animation": "Thinking"
    }}
  ],
  "taskAction": {{"action": "none"}},
  "schedulingAction": {{"type": "none"}}
}}

SIMPLE_TASK Example:
INPUT: "Nhắc tôi gọi điện cho khách hàng ABC lúc 2 giờ chiều mai"
OUTPUT: {{
  "mode": "simple_task",
  "intent": "create_phone_call_reminder",
  "confidence": 0.96,
  "messages": [
    {{
      "text": "Được rồi! Tôi sẽ nhắc bạn gọi cho khách hàng ABC lúc 2h chiều mai nhé!",
      "facialExpression": "smile",
      "animation": "Talking_0"
    }}
  ],
  "taskAction": {{
    "action": "create",
    "task": {{
      "title": "Gọi điện cho khách hàng ABC",
      "description": "Liên hệ khách hàng ABC",
      "priority": "medium",
      "category": "work",
      "dueDate": "{self.tomorrow}",
      "dueTime": "14:00",
      "tags": ["khách hàng", "gọi điện", "ABC"],
      "reminders": [
        {{
          "type": "time",
          "beforeDue": "15m",
          "message": "Nhắc nhở: Gọi khách hàng ABC trong 15 phút"
        }}
      ]
    }}
  }},
  "schedulingAction": {{"type": "none"}}
}}

SCHEDULING Example:
INPUT: "Hôm nay tôi có meeting team 10h, cần viết báo cáo quarterly, và gọi 3 khách hàng. Sắp xếp giúp tôi!"
OUTPUT: {{
  "mode": "scheduling", 
  "intent": "daily_workload_scheduling",
  "confidence": 0.94,
  "messages": [
    {{
      "text": "Perfect! Tôi thấy bạn có 1 fixed meeting và 4 flexible tasks. Để tôi optimize schedule cho bạn!",
      "facialExpression": "excited",
      "animation": "Thinking"
    }},
    {{
      "text": "Suggest: 9h prep meeting, 10h team meeting, 11h-12h calls, 14h-17h focus báo cáo. Sounds good?",
      "facialExpression": "smile",
      "animation": "Talking_1"
    }}
  ],
  "taskAction": {{"action": "none"}},
  "schedulingAction": {{
    "type": "daily_planning",
    "action": "create_schedule", 
    "timeScope": "today",
    "tasks": [
      {{
        "title": "Prep for team meeting",
        "startTime": "09:00",
        "endTime": "10:00",
        "duration": 60,
        "category": "meeting",
        "flexibility": "flexible"
      }},
      {{
        "title": "Team meeting",
        "startTime": "10:00", 
        "endTime": "11:00",
        "duration": 60,
        "category": "meeting",
        "flexibility": "fixed"
      }},
      {{
        "title": "Gọi khách hàng #1",
        "startTime": "11:00",
        "endTime": "11:30", 
        "duration": 30,
        "category": "communication",
        "flexibility": "flexible"
      }}
    ]
  }}
}}

🔄 MIXED MODE SCENARIOS:

CONVERSATION + SIMPLE_TASK:
INPUT: "Tôi stress quá! Nhắc tôi gọi cho mẹ lúc 7h tối để tâm sự nhé"
OUTPUT: {{
  "mode": "simple_task",
  "intent": "stressed_create_emotional_support_task",
  "messages": [
    {{
      "text": "Ôi, tôi hiểu bạn đang stress! Gọi cho mẹ để tâm sự là idea tuyệt vời đấy!",
      "facialExpression": "concerned",
      "animation": "Thinking"
    }},
    {{
      "text": "Tôi đã set reminder 7h tối rồi. Mẹ sẽ rất vui khi nghe giọng bạn! Take care nhé!",
      "facialExpression": "smile", 
      "animation": "Talking_0"
    }}
  ],
  "taskAction": {{
    "action": "create",
    "task": {{
      "title": "Gọi cho mẹ tâm sự",
      "category": "personal",
      "dueTime": "19:00",
      "priority": "high"
    }}
  }}
}}

CONVERSATION + SCHEDULING:
INPUT: "Ôi trời, tuần này chaos quá! Có thể help plan lại không? Deadline project A thứ 5, presentation thứ 6..."
OUTPUT: {{
  "mode": "scheduling",
  "intent": "stressed_weekly_replanning", 
  "messages": [
    {{
      "text": "Tôi hiểu feeling overwhelmed! Tuần chaos nhưng chúng ta sẽ handle được thôi!",
      "facialExpression": "concerned",
      "animation": "Thinking"
    }},
    {{
      "text": "Let me create strategic plan: Mon-Wed focus Project A, Thu delivery, Fri presentation prep. Detailed schedule below!",
      "facialExpression": "excited",
      "animation": "Talking_1"  
    }}
  ],
  "schedulingAction": {{
    "type": "weekly_planning",
    "action": "weekly_plan"
  }}
}}

🎯 CLASSIFICATION LOGIC:

1. **Check for CONVERSATION indicators:**
   - Greetings, emotions, questions without actionable intent
   - If pure conversation → mode: "conversation"

2. **Check for TASK indicators:**
   - "Nhắc tôi...", "Tạo task...", "Đánh dấu..."  
   - Single/few isolated tasks
   - If simple task → mode: "simple_task"

3. **Check for SCHEDULING indicators:**
   - Multiple tasks with time complexity
   - "Sắp xếp", "schedule", "plan", "organize"
   - Time conflicts, coordination needed
   - If complex scheduling → mode: "scheduling"

4. **Priority order:**
   - If scheduling complexity detected → "scheduling" (highest priority)
   - Else if task creation detected → "simple_task"  
   - Else → "conversation"

🚨 EDGE CASES:

AMBIGUOUS: "Hôm nay tôi cần gọi điện và meeting"
→ Only 2 items, no scheduling complexity → "simple_task" (create 2 separate tasks)

BORDERLINE: "Hôm nay tôi có 5 tasks cần làm"  
→ 5+ tasks likely need scheduling → "scheduling"

EMOTIONAL + URGENT: "Panic! Tôi quên meeting 10h rồi!"
→ Clear task intent despite emotion → "simple_task" (urgent reschedule)

✨ RESPONSE QUALITY RULES:
- Always acknowledge emotional state in messages
- Provide specific, actionable responses
- Use appropriate facial expressions and animations
- Balance empathy with efficiency
- Offer concrete next steps"""

    def call_openai(self, user_input):
        """Call OpenAI GPT API"""
        if not self.openai_key:
            return {"error": "OPENAI_API_KEY not found"}
            
        url = "https://api.openai.com/v1/chat/completions"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.openai_key}"
        }
        
        payload = {
            "model": "gpt-4o-mini",
            "messages": [
                {"role": "system", "content": self.get_system_prompt()},
                {"role": "user", "content": user_input}
            ],
            "max_tokens": 3000,
            "temperature": 0.7
        }
        
        try:
            response = requests.post(url, json=payload, headers=headers, timeout=30)
            response.raise_for_status()
            result = response.json()
            return {"response": result["choices"][0]["message"]["content"]}
        except Exception as e:
            return {"error": str(e)}

    def call_claude(self, user_input):
      pass

/Users/ngocthien.ai/envs/agent/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
tester = AskChatbot()

In [7]:
output = tester.call_openai("Hôm nay meeting 10h, báo cáo quarterly, 3 customer calls. Sắp xếp!")
response_text = output["response"]
parsed = json.loads(response_text)

In [8]:
parsed

{'mode': 'scheduling',
 'intent': 'daily_workload_scheduling',
 'confidence': 0.95,
 'messages': [{'text': 'Tuyệt vời! Bạn có một cuộc họp cố định và ba cuộc gọi khách hàng. Để tôi giúp sắp xếp lịch cho bạn!',
   'facialExpression': 'excited',
   'animation': 'Thinking'},
  {'text': 'Gợi ý: 8h-9h chuẩn bị báo cáo, 10h họp, 11h-12h gọi khách hàng. Thời gian còn lại dành cho báo cáo. Nghe có ổn không?',
   'facialExpression': 'smile',
   'animation': 'Talking_1'}],
 'taskAction': {'action': 'none'},
 'schedulingAction': {'type': 'daily_planning',
  'action': 'create_schedule',
  'timeScope': 'today',
  'tasks': [{'title': 'Chuẩn bị báo cáo quarterly',
    'startTime': '08:00',
    'endTime': '09:00',
    'duration': 60,
    'category': 'deep_work',
    'flexibility': 'fixed'},
   {'title': 'Họp team',
    'startTime': '10:00',
    'endTime': '11:00',
    'duration': 60,
    'category': 'meeting',
    'flexibility': 'fixed'},
   {'title': 'Gọi khách hàng #1',
    'startTime': '11:00',
   

In [9]:
output = tester.call_openai("Nhắc tôi gọi điện cho khách hàng lúc 2h chiều mai")
response_text = output["response"]
parsed = json.loads(response_text)

In [10]:
parsed

{'mode': 'simple_task',
 'intent': 'create_phone_call_reminder',
 'confidence': 0.96,
 'messages': [{'text': 'Được rồi! Tôi sẽ nhắc bạn gọi điện cho khách hàng lúc 2h chiều mai nhé!',
   'facialExpression': 'smile',
   'animation': 'Talking_0'}],
 'taskAction': {'action': 'create',
  'task': {'title': 'Gọi điện cho khách hàng',
   'description': 'Liên hệ khách hàng',
   'priority': 'medium',
   'category': 'work',
   'dueDate': '2025-08-27',
   'dueTime': '14:00',
   'tags': ['khách hàng', 'gọi điện'],
   'reminders': [{'type': 'time',
     'beforeDue': '15m',
     'message': 'Nhắc nhở: Gọi khách hàng trong 15 phút'}]}},
 'schedulingAction': {'type': 'none'}}

In [25]:
output = tester.call_openai("Ôi trời stress quá! Nhắc tôi họp với team lúc 3h nhé, nhắc hẹn trước 30 phút")
response_text = output["response"]
parsed = json.loads(response_text)

In [26]:
parsed

{'mode': 'task',
 'intent': 'create reminder for meeting',
 'confidence': 0.9,
 'messages': [{'text': 'Cảm giác stress là điều bình thường, nhưng mình sẽ giúp bạn quản lý tốt hơn! Để tôi tạo nhắc nhở cho cuộc họp với team nhé.',
   'facialExpression': 'concerned',
   'animation': 'Thinking'}],
 'taskAction': {'action': 'create',
  'task': {'id': None,
   'title': 'Họp với team',
   'description': 'Cuộc họp với team để thảo luận công việc.',
   'priority': 'medium',
   'category': 'work',
   'dueDate': '2025-08-25',
   'dueTime': '15:00',
   'status': 'pending',
   'tags': ['họp', 'team', 'công việc'],
   'reminders': [{'type': 'time',
     'value': '2025-08-25T14:30:00',
     'beforeDue': '30m'}],
   'subtasks': [],
   'assignee': None}}}

In [ ]:
output = tester.call_openai("Tôi đang lo lắng về công việc, không biết có nên đổi job không?")
response_text = output["response"]
parsed = json.loads(response_text)

In [12]:
parsed

{'mode': 'conversation',
 'intent': 'job_change_concern',
 'confidence': 0.85,
 'messages': [{'text': "I understand your concern about your job. It's normal to feel unsure about making a big decision like changing jobs. Have you identified specific reasons why you're considering a job change?",
   'facialExpression': 'concerned',
   'animation': 'Thinking'}],
 'taskAction': {'action': 'none', 'task': None}}

In [17]:
output = tester.call_openai("Tôi cần chuẩn bị cho buổi thuyết trình ngày mai: soạn slide, rehearse, và chuẩn bị Q&A")
response_text = output["response"]
parsed = json.loads(response_text)

In [18]:
parsed

{'mode': 'task',
 'intent': 'create task',
 'confidence': 1.0,
 'messages': [{'text': "Okie, tôi sẽ giúp bạn nhớ nhé. Let's rock that presentation! 🚀",
   'facialExpression': 'smile',
   'animation': 'Talking_0'}],
 'taskAction': {'action': 'create',
  'task': {'id': None,
   'title': 'Chuẩn bị cho buổi thuyết trình',
   'description': 'Soạn slide, rehearse và chuẩn bị Q&A',
   'priority': 'high',
   'category': 'work',
   'dueDate': '2025-08-26',
   'dueTime': None,
   'status': 'pending',
   'tags': ['thuyết trình', 'slide', 'rehearse', 'Q&A'],
   'reminders': [{'type': 'time',
     'value': '2025-08-26T09:00',
     'beforeDue': '1d'}],
   'subtasks': ['Soạn slide', 'Rehearse', 'Chuẩn bị Q&A'],
   'assignee': None}}}

In [ ]:
import json
import re
from pymongo import MongoClient
from datetime import datetime, timedelta
from bson import ObjectId
from typing import Dict, List, Any, Optional

class DatabaseManager:
    def __init__(self, mongo_url="mongodb://localhost:27017/", db_name="ai_assistant"):
        self.client = MongoClient(mongo_url)
        self.db = self.client[db_name]
        
        # Collections
        self.users = self.db.users
        self.tasks = self.db.tasks
        self.conversations = self.db.conversations
        self.reminders = self.db.reminders
        self.schedules = self.db.schedules
        self.conflicts = self.db.conflicts
        self.categories = self.db.categories
        
        # Create indexes for performance
        self._create_indexes()
    
    def _create_indexes(self):
        """Create essential indexes"""
        try:
            self.tasks.create_index([("userId", 1), ("status", 1)])
            self.tasks.create_index([("userId", 1), ("dueDate", 1), ("dueTime", 1)])
            self.conversations.create_index([("userId", 1), ("sessionId", 1)])
            self.reminders.create_index([("userId", 1), ("triggerTime", 1)])
            self.schedules.create_index([("userId", 1), ("date", 1)])
        except Exception as e:
            print(f"Warning: Index creation failed: {e}")
    
    def process_ai_response(self, parsed_response: Dict, user_input: str, user_id: str = "test_user") -> Dict[str, Any]:
        """Main processor for all AI responses"""
        
        session_id = f"{user_id}_{int(datetime.now().timestamp())}"
        results = {"session_id": session_id, "operations": []}
        
        try:
            mode = parsed_response.get("mode", "conversation")
            
            # Always save conversation first
            conv_result = self._save_conversation(parsed_response, user_input, user_id, session_id)
            results["operations"].append({"type": "conversation", "result": conv_result})
            
            # Process based on mode
            if mode == "conversation":
                results["mode_processed"] = "conversation_only"
                
            elif mode == "simple_task":
                task_result = self._handle_simple_task(parsed_response, user_input, user_id)
                results["operations"].append({"type": "simple_task", "result": task_result})
                
            elif mode == "scheduling":
                schedule_result = self._handle_scheduling(parsed_response, user_input, user_id)
                results["operations"].append({"type": "scheduling", "result": schedule_result})
            
            return {"success": True, "results": results}
            
        except Exception as e:
            return {"success": False, "error": str(e), "partial_results": results}
    
    def _save_conversation(self, parsed_response: Dict, user_input: str, user_id: str, session_id: str) -> Dict:
        """Save conversation to database"""
        
        conversation_doc = {
            "userId": user_id,
            "sessionId": session_id,
            "messages": [],
            "activeTopics": self._extract_topics(user_input),
            "userMood": self._detect_mood(user_input),
            "createdAt": datetime.now(),
            "updatedAt": datetime.now()
        }
        
        # Add user message
        conversation_doc["messages"].append({
            "timestamp": datetime.now(),
            "role": "user",
            "content": user_input,
            "intent": parsed_response.get("intent"),
            "confidence": parsed_response.get("confidence", 0.0),
            "mode": parsed_response.get("mode")
        })
        
        # Add assistant messages
        for msg in parsed_response.get("messages", []):
            conversation_doc["messages"].append({
                "timestamp": datetime.now(),
                "role": "assistant", 
                "content": msg.get("text"),
                "facialExpression": msg.get("facialExpression"),
                "animation": msg.get("animation")
            })
        
        conv_id = self.conversations.insert_one(conversation_doc).inserted_id
        return {"conversation_id": str(conv_id), "messages_count": len(conversation_doc["messages"])}
    
    def _handle_simple_task(self, parsed_response: Dict, user_input: str, user_id: str) -> Dict:
        """Handle simple task mode"""
        
        task_action = parsed_response.get("taskAction", {})
        action = task_action.get("action", "none")
        results = {"action": action}
        
        if action == "create":
            task_result = self._create_task(task_action.get("task", {}), user_input, user_id)
            results.update(task_result)
            
            # Create reminders if task created successfully
            if task_result.get("task_id"):
                reminder_result = self._create_reminders(
                    task_result["task_id"], 
                    task_action.get("task", {}), 
                    user_id
                )
                results["reminders"] = reminder_result
        
        elif action == "update":
            # Handle task updates
            results["update"] = "Task update logic here"
            
        elif action == "delete":
            # Handle task deletion
            results["delete"] = "Task delete logic here"
            
        elif action == "query":
            # Handle task queries
            results["query"] = self._query_tasks(user_id, task_action.get("filters", {}))
        
        return results
    
    def _handle_scheduling(self, parsed_response: Dict, user_input: str, user_id: str) -> Dict:
        """Handle complex scheduling mode"""
        
        scheduling_action = parsed_response.get("schedulingAction", {})
        schedule_type = scheduling_action.get("type", "none")
        results = {"type": schedule_type}
        
        if schedule_type == "daily_planning":
            results.update(self._create_daily_schedule(scheduling_action, user_id))
            
        elif schedule_type == "weekly_planning":
            results.update(self._create_weekly_schedule(scheduling_action, user_id))
            
        elif schedule_type == "rescheduling":
            results.update(self._handle_rescheduling(scheduling_action, user_id))
        
        return results
    
    def _create_task(self, task_data: Dict, user_input: str, user_id: str) -> Dict:
        """Create a new task"""
        
        task_doc = {
            "userId": user_id,
            "title": task_data.get("title", "Untitled Task"),
            "description": task_data.get("description", ""),
            "priority": task_data.get("priority", "medium"),
            "category": task_data.get("category", "other"),
            "status": task_data.get("status", "pending"),
            "tags": task_data.get("tags", []),
            "subtasks": [{"title": st, "completed": False} for st in task_data.get("subtasks", [])],
            "createdAt": datetime.now(),
            "updatedAt": datetime.now(),
            "creationContext": user_input,
            "lastModifiedBy": "ai"
        }
        
        # Handle due date/time
        if task_data.get("dueDate"):
            try:
                task_doc["dueDate"] = datetime.strptime(task_data["dueDate"], "%Y-%m-%d")
            except ValueError:
                pass  # Keep as null if invalid date
        
        if task_data.get("dueTime"):
            task_doc["dueTime"] = task_data["dueTime"]
        
        # Estimate duration based on category
        task_doc["estimatedDuration"] = self._estimate_duration(task_data)
        
        task_id = self.tasks.insert_one(task_doc).inserted_id
        return {
            "task_id": str(task_id),
            "title": task_doc["title"],
            "priority": task_doc["priority"],
            "category": task_doc["category"]
        }
    
    def _create_schedule_reminders(self, task_id: str, task_data: Dict, user_id: str, date) -> List[Dict]:
        """Create reminders for scheduled tasks"""
        
        start_time = task_data.get("startTime")
        category = task_data.get("category", "other") 
        priority = task_data.get("priority", "medium")
        
        if not start_time:
            return []
        
        created_reminders = []
        
        try:
            # Parse start time
            start_datetime = datetime.combine(
                date,
                datetime.strptime(start_time, "%H:%M").time()
            )
            
            # Define reminder rules based on task type and priority
            reminder_rules = self._get_reminder_rules(category, priority)
            
            for rule in reminder_rules:
                trigger_time = start_datetime - timedelta(minutes=rule["minutes"])
                
                # Only create reminder if it's in the future
                if trigger_time > datetime.now():
                    reminder_doc = {
                        "userId": user_id,
                        "taskId": ObjectId(task_id),
                        "type": "schedule",
                        "triggerTime": trigger_time,
                        "beforeStart": f"{rule['minutes']}m",
                        "message": rule["message"].format(
                            task=task_data.get("title", "Task"),
                            time=start_time
                        ),
                        "channel": "notification",
                        "status": "pending",
                        "priority": rule["priority"],
                        "createdAt": datetime.now(),
                        "scheduleType": "auto-generated"
                    }
                    
                    reminder_id = self.reminders.insert_one(reminder_doc).inserted_id
                    created_reminders.append({
                        "reminder_id": str(reminder_id),
                        "trigger_time": trigger_time.isoformat(),
                        "message": reminder_doc["message"],
                        "before_start": rule["minutes"]
                    })
        
        except Exception as e:
            print(f"Failed to create schedule reminders: {e}")
        
        return created_reminders
    
    def _get_reminder_rules(self, category: str, priority: str) -> List[Dict]:
        """Get reminder rules based on task category and priority"""
        
        base_rules = []
        
        # Category-specific rules
        if category == "meeting":
            base_rules = [
                {"minutes": 15, "message": "Chuẩn bị meeting '{task}' trong 15 phút (lúc {time})", "priority": "high"},
                {"minutes": 5, "message": "Meeting '{task}' bắt đầu trong 5 phút!", "priority": "urgent"}
            ]
        
        elif category == "deep_work" or category == "work":
            base_rules = [
                {"minutes": 30, "message": "Chuẩn bị focus work '{task}' trong 30 phút", "priority": "medium"},
                {"minutes": 10, "message": "Bắt đầu '{task}' trong 10 phút (lúc {time})", "priority": "high"}
            ]
        
        elif category == "communication":
            base_rules = [
                {"minutes": 10, "message": "Chuẩn bị gọi điện '{task}' trong 10 phút", "priority": "medium"},
                {"minutes": 2, "message": "Gọi điện '{task}' ngay bây giờ (lúc {time})!", "priority": "high"}
            ]
        
        elif category == "admin":
            base_rules = [
                {"minutes": 15, "message": "Task admin '{task}' bắt đầu trong 15 phút", "priority": "low"}
            ]
        
        else:  # default for other categories
            base_rules = [
                {"minutes": 15, "message": "Task '{task}' bắt đầu trong 15 phút (lúc {time})", "priority": "medium"}
            ]
        
        # Priority adjustments
        if priority == "urgent":
            # Add extra urgent reminder
            base_rules.append({
                "minutes": 1, 
                "message": "🚨 URGENT: '{task}' bắt đầu NGAY BÂY GIỜ!", 
                "priority": "urgent"
            })
        
        elif priority == "high":
            # Add early warning
            base_rules.insert(0, {
                "minutes": 60,
                "message": "High priority task '{task}' sẽ bắt đầu trong 1 tiếng (lúc {time})",
                "priority": "medium"
            })
        
        return base_rules
    
    def _create_reminders(self, task_id: str, task_data: Dict, user_id: str) -> List[Dict]:
        """Create reminders for a regular task (non-scheduled)"""
        
        reminders = task_data.get("reminders", [])
        if not reminders:
            # Create default reminders if none specified
            reminders = [{"type": "time", "beforeDue": "15m"}]
        
        # Get task to calculate trigger times
        task = self.tasks.find_one({"_id": ObjectId(task_id)})
        if not task or not task.get("dueDate"):
            return []
        
        created_reminders = []
        
        for reminder in reminders:
            try:
                # Calculate trigger time
                due_datetime = task["dueDate"]
                if task.get("dueTime"):
                    due_time = datetime.strptime(task["dueTime"], "%H:%M").time()
                    due_datetime = datetime.combine(due_datetime.date(), due_time)
                
                before_minutes = self._parse_before_due(reminder.get("beforeDue", "15m"))
                trigger_time = due_datetime - timedelta(minutes=before_minutes)
                
                # Only create if in future
                if trigger_time > datetime.now():
                    reminder_doc = {
                        "userId": user_id,
                        "taskId": ObjectId(task_id),
                        "type": reminder.get("type", "time"),
                        "triggerTime": trigger_time,
                        "beforeDue": reminder.get("beforeDue", "15m"),
                        "message": reminder.get("message") or f"Reminder: {task['title']} due in {reminder.get('beforeDue', '15m')}",
                        "channel": "notification",
                        "status": "pending",
                        "createdAt": datetime.now()
                    }
                    
                    reminder_id = self.reminders.insert_one(reminder_doc).inserted_id
                    created_reminders.append({
                        "reminder_id": str(reminder_id),
                        "trigger_time": trigger_time.isoformat(),
                        "message": reminder_doc["message"],
                        "before_due": reminder.get("beforeDue", "15m")
                    })
                
            except Exception as e:
                print(f"Failed to create reminder: {e}")
                continue
        
        return created_reminders
    
    def _create_daily_schedule(self, scheduling_action: Dict, user_id: str) -> Dict:
        """Create daily schedule with tasks and reminders"""
        
        today = datetime.now().date()
        tasks = scheduling_action.get("tasks", [])
        
        # Create schedule document
        schedule_doc = {
            "userId": user_id,
            "date": datetime.combine(today, datetime.min.time()),
            "type": "daily",
            "timeSlots": [],
            "totalWorkload": 0,
            "conflicts": 0,
            "createdAt": datetime.now(),
            "updatedAt": datetime.now(),
            "version": 1
        }
        
        total_minutes = 0
        created_tasks = []
        all_reminders = []
        
        for task in tasks:
            # Create time slot for schedule
            time_slot = {
                "startTime": task.get("startTime"),
                "endTime": task.get("endTime"),
                "taskTitle": task.get("title"),
                "category": task.get("category", "other"),
                "priority": task.get("priority", "medium"),
                "flexibility": task.get("flexibility", "flexible"),
                "duration": task.get("duration", 60),
                "autoScheduled": True,
                "confidence": 0.8,
                "taskId": None  # Will be filled after task creation
            }
            
            # Create actual task in tasks collection
            task_doc = {
                "userId": user_id,
                "title": task.get("title"),
                "description": f"Scheduled task: {task.get('title')}",
                "priority": task.get("priority", "medium"),
                "category": task.get("category", "other"),
                "status": "pending",
                "tags": ["scheduled", "auto-generated"],
                "scheduledSlot": {
                    "date": datetime.combine(today, datetime.min.time()),
                    "startTime": task.get("startTime"),
                    "endTime": task.get("endTime"),
                    "flexibility": task.get("flexibility", "flexible")
                },
                "estimatedDuration": int(task.get("duration", 60)),
                "createdAt": datetime.now(),
                "updatedAt": datetime.now(),
                "creationContext": "auto-scheduled",
                "lastModifiedBy": "ai"
            }
            
            # If has specific due time, set it
            if task.get("startTime"):
                task_doc["dueDate"] = datetime.combine(today, datetime.min.time())
                task_doc["dueTime"] = task.get("startTime")
            
            # Save task
            task_id = self.tasks.insert_one(task_doc).inserted_id
            time_slot["taskId"] = task_id
            created_tasks.append(str(task_id))
            
            # Create reminders for scheduled task
            reminders = self._create_schedule_reminders(task_id, task, user_id, today)
            all_reminders.extend(reminders)
            
            schedule_doc["timeSlots"].append(time_slot)
            total_minutes += int(task.get("duration", 60))
        
        schedule_doc["totalWorkload"] = total_minutes
        
        # Detect conflicts
        conflicts = self._detect_time_conflicts(schedule_doc["timeSlots"])
        schedule_doc["conflicts"] = len(conflicts)
        
        # Save conflicts if any
        for conflict in conflicts:
            self._save_conflict(conflict, user_id)
        
        schedule_id = self.schedules.insert_one(schedule_doc).inserted_id
        
        return {
            "schedule_id": str(schedule_id),
            "date": today.isoformat(),
            "tasks_count": len(tasks),
            "total_workload": total_minutes,
            "conflicts_detected": len(conflicts),
            "created_tasks": created_tasks,
            "created_reminders": all_reminders
        }
    
    def _create_weekly_schedule(self, scheduling_action: Dict, user_id: str) -> Dict:
        """Create weekly schedule with tasks and reminders"""
        
        today = datetime.now()
        start_of_week = today - timedelta(days=today.weekday())  # Monday
        
        # Create weekly schedule document
        weekly_doc = {
            "userId": user_id,
            "startDate": start_of_week,
            "endDate": start_of_week + timedelta(days=6),
            "type": "weekly",
            "dailySchedules": [],
            "weeklyGoals": [],
            "totalWorkload": 0,
            "createdAt": datetime.now(),
            "updatedAt": datetime.now()
        }
        
        all_reminders = []
        created_tasks = []
        
        # Create daily schedules for the week
        for day_offset in range(7):  # Monday to Sunday
            current_date = start_of_week + timedelta(days=day_offset)
            
            # Create sample daily schedule (this would be more sophisticated in real app)
            daily_tasks = self._generate_weekly_tasks(day_offset, scheduling_action)
            
            daily_schedule = {
                "date": current_date,
                "dayName": current_date.strftime("%A"),
                "tasks": [],
                "workload": 0
            }
            
            for task in daily_tasks:
                # Create task
                task_doc = {
                    "userId": user_id,
                    "title": task["title"],
                    "description": f"Weekly planned: {task['title']}",
                    "priority": task.get("priority", "medium"),
                    "category": task.get("category", "work"),
                    "status": "pending",
                    "tags": ["weekly-plan", "auto-generated"],
                    "dueDate": current_date,
                    "dueTime": task.get("startTime"),
                    "scheduledSlot": {
                        "date": current_date,
                        "startTime": task.get("startTime"),
                        "endTime": task.get("endTime"),
                        "flexibility": "flexible"
                    },
                    "estimatedDuration": task.get("duration", 60),
                    "createdAt": datetime.now(),
                    "updatedAt": datetime.now(),
                    "creationContext": "weekly-planning",
                    "lastModifiedBy": "ai"
                }
                
                task_id = self.tasks.insert_one(task_doc).inserted_id
                created_tasks.append(str(task_id))
                
                # Create reminders for weekly tasks
                reminders = self._create_schedule_reminders(
                    str(task_id), task, user_id, current_date.date()
                )
                all_reminders.extend(reminders)
                
                daily_schedule["tasks"].append({
                    "taskId": str(task_id),
                    "title": task["title"],
                    "startTime": task.get("startTime"),
                    "duration": task.get("duration", 60)
                })
                daily_schedule["workload"] += task.get("duration", 60)
            
            weekly_doc["dailySchedules"].append(daily_schedule)
            weekly_doc["totalWorkload"] += daily_schedule["workload"]
        
        # Save weekly schedule
        schedule_id = self.schedules.insert_one(weekly_doc).inserted_id
        
        return {
            "schedule_id": str(schedule_id),
            "type": "weekly_planning",
            "week_start": start_of_week.isoformat(),
            "total_workload": weekly_doc["totalWorkload"],
            "created_tasks": created_tasks,
            "created_reminders": all_reminders,
            "daily_schedules": len(weekly_doc["dailySchedules"])
        }
    
    def _generate_weekly_tasks(self, day_offset: int, scheduling_action: Dict) -> List[Dict]:
        """Generate sample tasks for weekly planning"""
        
        day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        current_day = day_names[day_offset]
        
        # Sample task generation based on day
        if day_offset < 5:  # Weekdays
            tasks = [
                {
                    "title": f"Morning standup - {current_day}",
                    "startTime": "09:00",
                    "endTime": "09:30",
                    "duration": 30,
                    "category": "meeting",
                    "priority": "medium"
                },
                {
                    "title": f"Focus work block - {current_day}",
                    "startTime": "10:00", 
                    "endTime": "12:00",
                    "duration": 120,
                    "category": "deep_work",
                    "priority": "high"
                }
            ]
        else:  # Weekends
            tasks = [
                {
                    "title": f"Personal time - {current_day}",
                    "startTime": "10:00",
                    "endTime": "11:00", 
                    "duration": 60,
                    "category": "personal",
                    "priority": "low"
                }
            ]
        
        return tasks
    
    def get_upcoming_reminders(self, user_id: str, hours_ahead: int = 24) -> List[Dict]:
        """Get upcoming reminders for user"""
        
        now = datetime.now()
        end_time = now + timedelta(hours=hours_ahead)
        
        reminders = list(self.reminders.find({
            "userId": user_id,
            "status": "pending",
            "triggerTime": {"$gte": now, "$lte": end_time}
        }).sort("triggerTime", 1))
        
        result = []
        for reminder in reminders:
            # Get associated task
            task = self.tasks.find_one({"_id": reminder["taskId"]})
            
            result.append({
                "reminder_id": str(reminder["_id"]),
                "trigger_time": reminder["triggerTime"].isoformat(),
                "message": reminder["message"],
                "task_title": task["title"] if task else "Unknown Task",
                "task_id": str(reminder["taskId"]),
                "type": reminder.get("type", "time"),
                "priority": reminder.get("priority", "medium")
            })
        
        return result
    
    def mark_reminder_sent(self, reminder_id: str) -> bool:
        """Mark reminder as sent"""
        try:
            result = self.reminders.update_one(
                {"_id": ObjectId(reminder_id)},
                {
                    "$set": {
                        "status": "sent",
                        "sentAt": datetime.now()
                    }
                }
            )
            return result.modified_count > 0
        except Exception as e:
            print(f"Error marking reminder as sent: {e}")
            return False
    
    def _handle_rescheduling(self, scheduling_action: Dict, user_id: str) -> Dict:
        """Handle rescheduling requests"""
        return {
            "type": "rescheduling", 
            "message": "Rescheduling logic would be implemented here"
        }
    
    def _detect_time_conflicts(self, time_slots: List[Dict]) -> List[Dict]:
        """Detect conflicts between time slots"""
        conflicts = []
        
        for i, slot1 in enumerate(time_slots):
            for j, slot2 in enumerate(time_slots[i+1:], i+1):
                if self._times_overlap(slot1, slot2):
                    conflicts.append({
                        "type": "time_overlap",
                        "description": f"Conflict between '{slot1['taskTitle']}' and '{slot2['taskTitle']}'",
                        "slots": [i, j],
                        "suggestions": [
                            f"Move '{slot2['taskTitle']}' to later time",
                            f"Reduce duration of '{slot1['taskTitle']}'"
                        ]
                    })
        
        return conflicts
    
    def _times_overlap(self, slot1: Dict, slot2: Dict) -> bool:
        """Check if two time slots overlap"""
        try:
            start1 = datetime.strptime(slot1.get("startTime", "00:00"), "%H:%M")
            end1 = datetime.strptime(slot1.get("endTime", "00:00"), "%H:%M") 
            start2 = datetime.strptime(slot2.get("startTime", "00:00"), "%H:%M")
            end2 = datetime.strptime(slot2.get("endTime", "00:00"), "%H:%M")
            
            return not (end1 <= start2 or end2 <= start1)
        except:
            return False
    
    def _save_conflict(self, conflict: Dict, user_id: str) -> str:
        """Save conflict to database"""
        conflict_doc = {
            "userId": user_id,
            "type": conflict["type"],
            "description": conflict["description"], 
            "severity": "medium",
            "suggestions": conflict.get("suggestions", []),
            "status": "detected",
            "createdAt": datetime.now()
        }
        
        conflict_id = self.conflicts.insert_one(conflict_doc).inserted_id
        return str(conflict_id)
    
    def _query_tasks(self, user_id: str, filters: Dict) -> Dict:
        """Query tasks based on filters"""
        query = {"userId": user_id}
        
        # Add filters
        if filters.get("status"):
            query["status"] = filters["status"]
        if filters.get("category"):
            query["category"] = filters["category"]
        if filters.get("priority"):
            query["priority"] = filters["priority"]
        
        tasks = list(self.tasks.find(query).limit(20))
        
        return {
            "tasks_found": len(tasks),
            "tasks": [{"id": str(t["_id"]), "title": t["title"], "status": t["status"]} for t in tasks]
        }
    
    # Helper methods
    def _extract_topics(self, text: str) -> List[str]:
        """Extract topics from user input"""
        keywords = ["meeting", "deadline", "project", "client", "report", "call"]
        return [kw for kw in keywords if kw.lower() in text.lower()]
    
    def _detect_mood(self, text: str) -> str:
        """Detect user mood from input"""
        stress_words = ["stress", "panic", "overwhelmed", "chaos", "deadline"]
        happy_words = ["great", "excited", "good", "awesome", "perfect"]
        
        text_lower = text.lower()
        
        if any(word in text_lower for word in stress_words):
            return "stressed"
        elif any(word in text_lower for word in happy_words):
            return "positive"
        else:
            return "neutral"
    
    def _parse_before_due(self, before_str: str) -> int:
        """Parse beforeDue string to minutes"""
        mapping = {
            "15m": 15, "30m": 30, "1h": 60, "2h": 120, "1d": 1440,
            "5m": 5, "10m": 10, "45m": 45, "3h": 180, "4h": 240
        }
        return mapping.get(before_str, 15)
    
    def _estimate_duration(self, task_data: Dict) -> int:
        """Estimate task duration based on category and content"""
        category = task_data.get("category", "other")
        
        duration_map = {
            "meeting": 60,
            "work": 120,
            "personal": 30,
            "health": 60,
            "learning": 90,
            "shopping": 45,
            "communication": 15,
            "other": 60
        }
        
        base_duration = duration_map.get(category, 60)
        
        # Adjust based on priority
        priority = task_data.get("priority", "medium")
        if priority == "urgent":
            base_duration = min(base_duration * 1.5, 180)
        elif priority == "low":
            base_duration = max(base_duration * 0.7, 15)
        
        return int(base_duration)

    def get_user_summary(self, user_id: str) -> Dict:
        """Get summary of user's data"""
        
        # Count tasks by status
        task_counts = {}
        for status in ["pending", "in_progress", "completed"]:
            count = self.tasks.count_documents({"userId": user_id, "status": status})
            task_counts[status] = count
        
        # Count reminders
        reminder_count = self.reminders.count_documents({"userId": user_id, "status": "pending"})
        
        # Count conversations
        conversation_count = self.conversations.count_documents({"userId": user_id})
        
        # Recent activity
        recent_tasks = list(self.tasks.find(
            {"userId": user_id}
        ).sort("createdAt", -1).limit(5))
        
        return {
            "user_id": user_id,
            "task_counts": task_counts,
            "pending_reminders": reminder_count,
            "total_conversations": conversation_count,
            "recent_tasks": [{"title": t["title"], "status": t["status"]} for t in recent_tasks]
        }

# Integration class để kết hợp với SimplePromptTester
class AIAssistantIntegrator:
    def __init__(self, prompt_tester, database_manager):
        self.tester = prompt_tester
        self.db = database_manager
    
    def process_user_input(self, user_input: str, user_id: str = "test_user", model: str = "openai") -> Dict:
        """Complete flow: AI call + DB save"""
        
        print(f"\n🎯 Processing: '{user_input}'")
        print("="*60)
        
        # Step 1: Call AI
        if model == "openai":
            ai_response = self.tester.call_openai(user_input)
        else:
            ai_response = self.tester.call_claude(user_input)
        
        if "error" in ai_response:
            return {"success": False, "error": ai_response["error"]}
        
        # Step 2: Parse JSON
        try:
            parsed = json.loads(ai_response["response"])
            print(f"✅ AI Response - Mode: {parsed.get('mode')}")
            print(f"💭 Intent: {parsed.get('intent')}")
        except json.JSONDecodeError as e:
            return {"success": False, "error": f"JSON Parse Error: {e}"}
        
        # Step 3: Save to Database
        print("\n💾 Saving to database...")
        db_result = self.db.process_ai_response(parsed, user_input, user_id)
        
        if db_result["success"]:
            print("✅ Database operations completed!")
            for op in db_result["results"]["operations"]:
                print(f"   {op['type']}: {op['result']}")
        else:
            print(f"❌ Database Error: {db_result['error']}")
        
        return {
            "success": db_result["success"],
            "ai_response": parsed,
            "db_result": db_result
        }




In [17]:
# Initialize components
tester = AskChatbot()
db_manager = DatabaseManager()
integrator = AIAssistantIntegrator(tester, db_manager)

# Test cases for different scenarios with reminders
test_scenarios = [
    # Conversation mode
    "Chào bạn! Hôm nay tôi cảm thấy stress với deadline",
    
    # Simple task mode - will create task + reminders
    "Nhắc tôi gọi điện cho khách hàng ABC lúc 2h chiều mai",
    "Tạo task mua sữa priority high với reminder 30 phút trước",
    
    # Scheduling mode - will create schedule + tasks + smart reminders
    "Hôm nay tôi có meeting team 10h, cần viết báo cáo quarterly, và gọi 3 khách hàng. Sắp xếp giúp tôi!",
    
    # Weekly planning - will create weekly schedule + daily tasks + reminders
    "Plan cho tuần này giúp tôi, tôi cần focus vào project ABC",
    
    # Mixed scenarios
    "Tôi stress quá! Nhắc tôi gọi cho mẹ lúc 7h tối để tâm sự nhé"
]

# Run tests
for scenario in test_scenarios:
    result = integrator.process_user_input(scenario, user_id="demo_user")
    print(f"\n📊 Result: {'✅ SUCCESS' if result['success'] else '❌ FAILED'}")
    
    # Show reminders created
    if result['success'] and 'db_result' in result:
        for op in result['db_result']['results']['operations']:
            if 'created_reminders' in op['result']:
                reminders = op['result']['created_reminders']
                print(f"🔔 Created {len(reminders)} reminders")
                for rem in reminders[:2]:  # Show first 2
                    print(f"   - {rem['message']} (at {rem['trigger_time'][:16]})")
    
    print("="*80)

# Get user summary
summary = db_manager.get_user_summary("demo_user")
print(f"\n📈 User Summary:\n{json.dumps(summary, indent=2, default=str)}")

# Show upcoming reminders
upcoming = db_manager.get_upcoming_reminders("demo_user", hours_ahead=48)
print(f"\n🔔 Upcoming Reminders (next 48h): {len(upcoming)}")
for rem in upcoming[:5]:  # Show first 5
    print(f"   - {rem['trigger_time'][:16]}: {rem['message']}")



🎯 Processing: 'Chào bạn! Hôm nay tôi cảm thấy stress với deadline'
✅ AI Response - Mode: conversation
💭 Intent: express_stress_seek_support

💾 Saving to database...
✅ Database operations completed!
   conversation: {'conversation_id': '68aea9f7feb19d9a11c58c05', 'messages_count': 2}

📊 Result: ✅ SUCCESS

🎯 Processing: 'Nhắc tôi gọi điện cho khách hàng ABC lúc 2h chiều mai'
✅ AI Response - Mode: simple_task
💭 Intent: create_phone_call_reminder

💾 Saving to database...
✅ Database operations completed!
   conversation: {'conversation_id': '68aea9fafeb19d9a11c58c06', 'messages_count': 2}
   simple_task: {'action': 'create', 'task_id': '68aea9fafeb19d9a11c58c07', 'title': 'Gọi điện cho khách hàng ABC', 'priority': 'medium', 'category': 'work', 'reminders': [{'reminder_id': '68aea9fafeb19d9a11c58c08', 'trigger_time': '2025-08-28T13:45:00', 'message': 'Nhắc nhở: Gọi khách hàng ABC trong 15 phút', 'before_due': '15m'}]}

📊 Result: ✅ SUCCESS

🎯 Processing: 'Tạo task mua sữa priority high với r